# 12: Local LLM + MCP (บทเสริม)
    แบบฝึกหัดสำหรับผู้เข้าร่วม Sri Trang

    เลือก Python kernel แล้วกด **Run All** cell แรกจะติดตั้ง packages ให้
    สำหรับการรัน local ให้เปิด Ollama ที่มีโมเดลตาม README ไว้ก่อน
    โค้ด HTTP API ใช้ได้ทั้ง local และ Colab แต่ `localhost` ใน Colab หมายถึงเครื่อง Colab
    หากเรียก Ollama จาก Colab ต้องกำหนด HOST เป็น endpoint ที่ runtime เข้าถึงได้
    โค้ดและข้อมูลทั้งหมดอยู่ใน cells ไม่ต้องเตรียมหรือแก้ไฟล์ประกอบ
    ตัวอย่างทั้งหมดเป็นข้อมูลสมมติสำหรับการเรียน ไม่ใช่ข้อมูลหรือ SOP จริงของบริษัท


## ติดตั้ง packages
ใช้ Python kernel ที่รองรับ async cells ทั้ง local Jupyter และ Colab


In [ ]:
%pip install -q "requests>=2.32,<3" "mcp>=1.10,<2"


## 1. เชื่อมต่อ local model
ใช้โมเดลที่รองรับ tool calling เพื่อเลือกเครื่องมือจาก MCP


In [ ]:
import requests

HOST = "http://localhost:11434"
MODEL = "qwen3:4b"

def chat(messages, **kwargs):
    payload = {"model": MODEL, "messages": messages, "stream": False,
               "think": False, "options": {"temperature": 0}, **kwargs}
    response = requests.post(HOST + "/api/chat", json=payload, timeout=180)
    return response.json()["message"]


## 2. เตรียม MCP server ใน cell
เครื่องมืออ่านสถานะรับน้ำยางของโรงงานสมมติ A/B โค้ด server อยู่ที่นี่และรันเป็น subprocess โดยไม่สร้างไฟล์


In [ ]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_code = """
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Factory workshop", log_level="CRITICAL")

@mcp.tool()
def get_factory_status(factory_id: str) -> dict:
    \"\"\"Get fictional latex receiving and QC status for factory A or B.\"\"\"
    factories = {
        "A": {"received_tonnes": 12, "qc_status": "pending", "lot_id": "L-101"},
        "B": {"received_tonnes": 8, "qc_status": "completed", "lot_id": "L-102"},
    }
    return {"factory_id": factory_id, "data_type": "fictional",
            **factories.get(factory_id, {"status": "unknown"})}

mcp.run(transport="stdio")
"""
server_parameters = StdioServerParameters(command=sys.executable, args=["-c", server_code])


## 3. อ่านเครื่องมือจาก MCP
แปลงชื่อ คำอธิบาย และ input schema เป็นรูปแบบที่ Ollama รับได้


In [ ]:
async def get_tools():
    async with stdio_client(server_parameters) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            discovered = await session.list_tools()
            return [{"type": "function", "function": {
                "name": tool.name, "description": tool.description,
                "parameters": tool.inputSchema,
            }} for tool in discovered.tools]

tools = await get_tools()


## 4. ให้โมเดลเลือกเครื่องมือ
คำถามขอข้อมูลโรงงาน A ซึ่งโมเดลต้องอ่านจากเครื่องมือ


In [ ]:
question = "ใช้ get_factory_status โดย factory_id เป็น A แล้วสรุปสถานะรับน้ำยางภาษาไทย ระบุว่าเป็นข้อมูลสมมติ"
messages = [
    {"role": "system", "content": "เรียกเครื่องมือหนึ่งครั้งเพื่อตรวจสถานะโรงงานก่อนตอบ"},
    {"role": "user", "content": question},
]
selected = chat(messages, tools=tools)
messages.append(selected)


## 5. เรียกเครื่องมือผ่าน MCP แล้วสรุปคำตอบ
Notebook ส่งคำขอไปยัง MCP server และนำผลกลับให้ local LLM จากนั้นปิด server อัตโนมัติ


In [ ]:
async def execute_tools():
    async with stdio_client(server_parameters) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            for call in selected.get("tool_calls", []):
                function = call["function"]
                result = await session.call_tool(function["name"], function["arguments"])
                tool_text = "\n".join(block.text for block in result.content if block.type == "text")
                messages.append({"role": "tool", "tool_name": function["name"], "content": tool_text})

await execute_tools()
answer = chat(messages)
print(answer["content"])


## เสร็จแล้ว
ลองเปลี่ยนโรงงานเป็น B ในขั้นตอน 4 แล้วรันขั้นตอน 4–5 อีกครั้ง

โมเดลเลือกเครื่องมือ ส่วน notebook เรียกเครื่องมือผ่าน MCP หากเปลี่ยนโมเดลที่รองรับ tools สามารถใช้ MCP server เดิมได้
